In [1]:
import pandas as pd
import os

In [2]:
# Check available test CSV files
test_files = [
    "test_0_49.csv",
    "test_50_99.csv", 
    "test_100_149.csv",
    "test_150_199.csv"
]

print("Available test CSV files:")
for file in test_files:
    if os.path.exists(file):
        df = pd.read_csv(file)
        print(f"{file}: {df.shape[0]} rows, {df.shape[1]} columns")
    else:
        print(f"{file}: File not found")
print()

Available test CSV files:
test_0_49.csv: 200 rows, 10 columns
test_50_99.csv: 200 rows, 10 columns
test_100_149.csv: 200 rows, 10 columns
test_150_199.csv: 200 rows, 10 columns



In [3]:
# Load base dataset from first file and merge better_english_translation from all shards
test_files_with_ranges = [
    ("test_0_49.csv", 0, 49),
    ("test_50_99.csv", 50, 99), 
    ("test_100_149.csv", 100, 149),
    ("test_150_199.csv", 150, 199)
]

# Start with the first file as base dataset
base_file = test_files_with_ranges[0][0]
if os.path.exists(base_file):
    merged_df = pd.read_csv(base_file)
    print(f"Base dataset loaded from {base_file}: {merged_df.shape[0]} rows, {merged_df.shape[1]} columns")
    
    # Ensure better_english_translation column exists
    if 'better_english_translation' not in merged_df.columns:
        merged_df['better_english_translation'] = None
        print("Added better_english_translation column")
    
    # Now merge better_english_translation from other shards
    for filename, start_idx, end_idx in test_files_with_ranges[1:]:
        if os.path.exists(filename):
            shard_df = pd.read_csv(filename)
            print(f"Processing {filename} for rows {start_idx}-{end_idx}")
            
            # Copy better_english_translation for the specific range
            if 'better_english_translation' in shard_df.columns:
                better_translations = shard_df.loc[start_idx:end_idx, 'better_english_translation']
                valid_translations = better_translations.notna()
                
                print(f"  - Found {valid_translations.sum()} better translations in range {start_idx}-{end_idx}")
                
                # Update the merged dataset
                merged_df.loc[start_idx:end_idx, 'better_english_translation'] = better_translations
            else:
                print(f"  - No better_english_translation column found in {filename}")
        else:
            print(f"Warning: {filename} not found, skipping...")
    
    print(f"\nFinal merged dataset: {merged_df.shape[0]} rows, {merged_df.shape[1]} columns")
else:
    print(f"❌ Base file {base_file} not found!")
    merged_df = None
print()

Base dataset loaded from test_0_49.csv: 200 rows, 10 columns
Processing test_50_99.csv for rows 50-99
  - Found 50 better translations in range 50-99
Processing test_100_149.csv for rows 100-149
  - Found 50 better translations in range 100-149
Processing test_150_199.csv for rows 150-199
  - Found 50 better translations in range 150-199

Final merged dataset: 200 rows, 10 columns



In [4]:
# Check the merge results and data quality
if merged_df is not None:
    print("Data Quality Check:")
    print(f"Total rows: {len(merged_df)}")
    
    # Check column information
    print(f"\nColumns: {list(merged_df.columns)}")
    
    # Check translation coverage
    if 'english_translation' in merged_df.columns:
        eng_trans_count = merged_df['english_translation'].notna().sum()
        print(f"Rows with english_translation: {eng_trans_count}")
    
    if 'better_english_translation' in merged_df.columns:
        better_trans_count = merged_df['better_english_translation'].notna().sum()
        print(f"Rows with better_english_translation: {better_trans_count}")
        
        # Show sample of better translations from each range
        print(f"\nSample better translations by range:")
        ranges = [(0, 49), (50, 99), (100, 149), (150, 199)]
        for start, end in ranges:
            if end < len(merged_df):
                range_translations = merged_df.loc[start:end, 'better_english_translation'].notna().sum()
                if range_translations > 0:
                    sample_idx = merged_df.loc[start:end][merged_df.loc[start:end, 'better_english_translation'].notna()].index[0]
                    sample_text = merged_df.loc[sample_idx, 'better_english_translation']
                    print(f"  Range {start}-{end}: {range_translations} translations (sample: '{sample_text[:50]}...')")
                else:
                    print(f"  Range {start}-{end}: {range_translations} translations")
    
    print(f"\nFirst few rows:")
    print(merged_df[['text', 'english_translation', 'better_english_translation']].head() if 'text' in merged_df.columns else merged_df.head())
else:
    print("No merged data available.")

Data Quality Check:
Total rows: 200

Columns: ['id', 'datetime', 'location', 'text', 'rating', 'accessibility', 'facility', 'activity', 'english_translation', 'better_english_translation']
Rows with english_translation: 200
Rows with better_english_translation: 200

Sample better translations by range:
  Range 0-49: 50 translations (sample: 'Even with Lembang's already crowded atmosphere, it...')
  Range 50-99: 50 translations (sample: 'There are so many photo spots. The afternoon and e...')
  Range 100-149: 50 translations (sample: 'Truly beautiful, especially how clean it is. The r...')
  Range 150-199: 50 translations (sample: 'A photo spot with a Bandung-style world heritage s...')

First few rows:
                                                text  \
0  dengan suasana lembang yang sudah padat masih ...   
1  supaya gk bingung pemberhentian di warung batu...   
2  tempatnya asri adem cocok bawa main anak-anak ...   
3  kebun bunganya indah unik ditambah patung-patu...   
4  asyik

In [5]:
# Save the merged dataset with better_english_translation from all shards
if merged_df is not None:
    # Save the merged dataset
    output_filename = "test_merged.csv"
    merged_df.to_csv(output_filename, index=False)
    print(f"✅ Merged test dataset saved as '{output_filename}'")
    print(f"Final dataset: {len(merged_df)} rows, {len(merged_df.columns)} columns")
    
    # Verify the saved file
    verification_df = pd.read_csv(output_filename)
    print(f"✅ Verification: Saved file contains {len(verification_df)} rows")
    
    # Show detailed summary statistics
    print(f"\nDetailed Summary:")
    print(f"- Total rows: {len(merged_df)}")
    
    if 'text' in merged_df.columns:
        print(f"- Non-empty text entries: {merged_df['text'].notna().sum()}")
    
    if 'english_translation' in merged_df.columns:
        eng_count = merged_df['english_translation'].notna().sum()
        print(f"- Rows with english_translation: {eng_count}")
    
    if 'better_english_translation' in merged_df.columns:
        better_count = merged_df['better_english_translation'].notna().sum()
        print(f"- Rows with better_english_translation: {better_count}")
        
        # Breakdown by ranges
        print(f"\nBetter translation breakdown by ranges:")
        ranges = [(0, 49), (50, 99), (100, 149), (150, 199)]
        total_better = 0
        for start, end in ranges:
            if end < len(merged_df):
                range_count = merged_df.loc[start:end, 'better_english_translation'].notna().sum()
                total_better += range_count
                print(f"  Rows {start}-{end}: {range_count} better translations")
        print(f"  Total better translations: {total_better}")
        
else:
    print("❌ No data to save. Please check that test files exist and were loaded successfully.")

✅ Merged test dataset saved as 'test_merged.csv'
Final dataset: 200 rows, 10 columns
✅ Verification: Saved file contains 200 rows

Detailed Summary:
- Total rows: 200
- Non-empty text entries: 200
- Rows with english_translation: 200
- Rows with better_english_translation: 200

Better translation breakdown by ranges:
  Rows 0-49: 50 better translations
  Rows 50-99: 50 better translations
  Rows 100-149: 50 better translations
  Rows 150-199: 50 better translations
  Total better translations: 200


In [7]:
# Display final merged dataset info
if 'merged_df' in locals():
    print("Final Merged Dataset Preview:")
    print("="*50)
    merged_df.head(10)
else:
    print("No merged dataset available to display.")

Final Merged Dataset Preview:
